# Basic LLM calls: OpenAI first, other providers by adapter

## Learning goals

- Create the standard OpenAI client without repeating environment configuration
- Understand Responses API versus Chat Completions API
- Make one small text-generation helper that remains testable
- Know when an OpenAI-compatible `base_url` is appropriate
- Compare native calls for Anthropic, Gemini, and Groq with an OpenRouter-compatible call


## The short answer

Yes—when this notebook is calling OpenAI directly, the client should normally be created with:

```python
client = OpenAI()
```

The OpenAI Python SDK automatically reads `OPENAI_API_KEY`. It also honors `OPENAI_BASE_URL` when that variable is set, so leave `OPENAI_BASE_URL` empty or unset for the normal OpenAI service. There is no benefit in manually writing `api_key=os.getenv(...)` for the standard case.

Create one client and reuse it. Rebuilding the client inside every helper call is unnecessary and prevents the underlying HTTP client from efficiently reusing connections. Passing the client into our helper keeps the function easy to test.


## Install the SDK

For a new project:

```text
uv add openai
```

For this existing course repository, `openai` is already declared, so use `uv sync` after cloning.


## Load the project environment

A notebook's working directory can vary between editors. Instead of assuming that the kernel starts in a particular folder, walk upward until we find `pyproject.toml`, then load the `.env` beside it. `override=False` preserves values supplied by the terminal or deployment platform.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml above the working directory.")


project_root = find_project_root()
dotenv_path = project_root / ".env"
dotenv_loaded = load_dotenv(dotenv_path=dotenv_path, override=False)

print("Project root:", project_root)
print(".env file found:", dotenv_loaded)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))


## Mental model

```mermaid
flowchart LR
  A[Prompt] --> B[Small application helper]
  C[Validated provider config] --> B
  B --> D[Reusable provider client]
  D --> E{Provider API surface}
  E -->|OpenAI: recommended for new work| F[Responses API]
  E -->|Compatibility or existing apps| G[Chat Completions API]
  F --> H[response.output_text]
  G --> I[choices 0 message content]
  H --> J[Application string]
  I --> J
```

The helper owns the application-facing contract: text goes in and text comes out. The provider client owns authentication, HTTP transport, retries, and response types. The API surface determines how requests are shaped and where generated text is found.

This separation matters later. Agent code should not care whether OpenAI returns `response.output_text`, Anthropic returns content blocks, Gemini returns `response.text`, or a compatible Chat Completions provider returns `choices[0].message.content`. A thin provider adapter can normalize each result to a string.


## Responses API or Chat Completions?

OpenAI currently recommends the **Responses API for new projects**. Chat Completions remains supported. For an agentic-AI course, Responses is the better primary path because its design supports typed output items, built-in tools, function calls, multimodal input, and easier continuation through response IDs.

| Question | Responses API | Chat Completions API |
|---|---|---|
| Python call | `client.responses.create(...)` | `client.chat.completions.create(...)` |
| Input | A string or typed input items | A list of role-based messages |
| Text extraction | `response.output_text` convenience property | `completion.choices[0].message.content` |
| Agentic tools and state | First-class OpenAI path | Supported, but with the older message-oriented shape |
| Cross-provider compatibility | Mostly OpenAI-specific | Commonly emulated by OpenAI-compatible providers |
| Best fit here | New OpenAI and agentic examples | Learning the widely used chat shape or targeting a compatible gateway |

> The method is `client.chat.completions.create(...)`—plural `completions`. `response.create` and `chat.completions.create` are two different API surfaces, not two ways to read the same response.


## Recommended OpenAI implementation

The model name remains configuration because availability, quality, latency, and cost requirements differ. The course keeps its current fallback model while allowing `.env` to select another model. `store=False` makes retention intent explicit for this stateless example.


In [ ]:
from openai import OpenAI

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")


def call_openai_responses(
    prompt: str,
    *,
    client: OpenAI,
    model: str = OPENAI_MODEL,
) -> str:
    response = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=200,
        store=False,
    )
    return response.output_text


print("Responses helper defined. Model:", OPENAI_MODEL)


### Optional live call

The client creation is intentionally simple. Change `RUN_LIVE_CALL` to `True` only when your key is configured and you want to make a billable network request.


In [ ]:
RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before enabling the live call.")

    client = OpenAI()
    answer = call_openai_responses(
        "Explain an AI agent in one sentence.",
        client=client,
    )
    print(answer)
else:
    print("Live call skipped. Set RUN_LIVE_CALL = True when ready.")


## Equivalent Chat Completions example

Use this shape when maintaining an existing Chat Completions application or when a provider exposes an OpenAI-compatible chat endpoint but not the Responses API. Notice the different input and output shapes.


In [ ]:
def call_openai_chat_completions(
    prompt: str,
    *,
    client: OpenAI,
    model: str = OPENAI_MODEL,
) -> str:
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=200,
        store=False,
    )
    return completion.choices[0].message.content or ""


print("Chat Completions helper defined for comparison.")


## Test without making a network request

Passing the client into the helper gives us a clean seam for a fake. This test proves our application logic reads `output_text` correctly without needing an API key, spending tokens, or depending on network availability.


In [ ]:
from types import SimpleNamespace


class FakeResponses:
    def create(self, **request):
        assert request["input"] == "Hello"
        return SimpleNamespace(output_text="Hello from the fake model!")


fake_client = SimpleNamespace(responses=FakeResponses())
result = call_openai_responses("Hello", client=fake_client)

assert result == "Hello from the fake model!"
print(result)


## When should we pass `base_url`?

Use `base_url` only when intentionally calling a service that implements an OpenAI-compatible API. Make that provider choice explicit so a generic environment variable cannot silently redirect normal OpenAI traffic. Use that provider's own API key variable rather than reusing `OPENAI_API_KEY`.

```python
openrouter_client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)
```

An OpenAI-compatible endpoint usually supports the Chat Completions shape, but compatibility is not a promise that every OpenAI feature—including Responses, hosted tools, reasoning options, or response fields—is implemented. Verify the selected provider's documentation.


## Examples for other providers

Prefer a provider's native SDK when you need its full feature set and native response types. Use an OpenAI-compatible endpoint when portability and a common Chat Completions shape are more important. Keep these optional SDKs out of the core course environment until you actually use them.

### Anthropic: native SDK

```text
uv add anthropic
```

```python
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY
message = client.messages.create(
    model="your-anthropic-model",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt}],
)
text = "".join(block.text for block in message.content if block.type == "text")
```

### Gemini: native Google Gen AI SDK

```text
uv add google-genai
```

```python
from google import genai

client = genai.Client()  # reads GEMINI_API_KEY
response = client.models.generate_content(
    model="your-gemini-model",
    contents=prompt,
)
text = response.text
```

### Groq: native SDK

```text
uv add groq
```

```python
from groq import Groq

client = Groq()  # reads GROQ_API_KEY
completion = client.chat.completions.create(
    model="your-groq-model",
    messages=[{"role": "user", "content": prompt}],
)
text = completion.choices[0].message.content or ""
```

Groq also documents an OpenAI-compatible base URL: `https://api.groq.com/openai/v1`.

### OpenRouter: OpenAI-compatible SDK path

```python
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)
completion = client.chat.completions.create(
    model="provider/model-name",
    messages=[{"role": "user", "content": prompt}],
)
text = completion.choices[0].message.content or ""
```

> Model IDs change and differ by account or provider. Copy a currently available model ID from the provider's model catalog rather than baking an arbitrary cross-provider default into shared code.


## Best-practice provider design

Do not scatter `if provider == ...` checks across notebooks and agent logic. Give each provider a tiny adapter with the same application-facing method, such as `generate_text(prompt) -> str`. The adapter performs the provider-specific request and text extraction.

```mermaid
flowchart TD
  A[Agent or application] --> B[generate_text prompt]
  B --> C{Selected adapter}
  C --> D[OpenAI Responses]
  C --> E[Anthropic Messages]
  C --> F[Gemini generate_content]
  C --> G[Groq or OpenRouter Chat Completions]
  D --> H[Normalized text]
  E --> H
  F --> H
  G --> H
```

This preserves native capabilities behind each adapter while giving the rest of the application one stable contract. A production abstraction may also normalize usage, finish reasons, tool calls, errors, and streaming events—but plain text is enough for this first-principles notebook.


## Common failure cases

| Symptom | Likely cause | Response |
|---|---|---|
| `ModuleNotFoundError: openai` | Dependencies or kernel are out of sync | Run `uv sync` and select **Agentic AI Engineering** |
| Missing API-key error | `.env` was not found or the key is empty | Check the reported project root and configure the key without printing it |
| Authentication error | Wrong, revoked, or provider-mismatched key | Verify the credential belongs to the selected provider |
| Model-not-found error | Model ID is unavailable or misspelled | Check the provider's current model catalog |
| 404 or unsupported field on a compatible provider | It does not implement that OpenAI API surface or parameter | Use its documented Chat Completions shape or native SDK |
| Rate-limit or transient server error | Request limits or provider availability | Use bounded retries with backoff; do not retry invalid requests forever |

SDK exceptions contain useful status and request information. Surface those diagnostics, but never add API keys, authorization headers, or the full `.env` contents to logs.


## Exercise

1. Run the fake-client test and explain why it does not need an API key.
2. Enable the optional OpenAI live call and compare its output with the fake.
3. Write a fake for `call_openai_chat_completions` with the nested `choices[0].message.content` shape.
4. Sketch a `TextProvider` protocol with one method: `generate_text(prompt: str) -> str`.
5. Choose one additional provider and implement its adapter in a separate notebook without changing the agent-facing protocol.


## Official references

- [OpenAI: Migrate to the Responses API](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- [OpenAI: Responses API reference](https://developers.openai.com/api/docs/api-reference/responses)
- [OpenAI: Chat Completions API reference](https://developers.openai.com/api/docs/api-reference/chat)
- [Anthropic: Python SDK and libraries](https://platform.claude.com/docs/en/cli-sdks-libraries/overview)
- [Gemini: Text generation](https://ai.google.dev/gemini-api/docs/text-generation)
- [Groq: OpenAI compatibility](https://console.groq.com/docs/openai)
- [OpenRouter: Quickstart](https://openrouter.ai/docs/quickstart)


## Summary

For direct OpenAI use, create one reusable `OpenAI()` client and prefer the Responses API for new agentic work. Keep Chat Completions in your toolkit because it remains supported and is the most common compatibility surface across providers. For Anthropic, Gemini, and Groq, prefer native SDKs when provider-specific capabilities matter; use an explicit OpenAI-compatible client for services such as OpenRouter. Hide these differences behind small adapters so application and agent code receive one predictable result.
